# goal

- 5 Raw Tables
-        ↓
- Snapshot Grid
- (customer × month)
-         ↓
- Aggregate features from each table
-         ↓
- Join features together
-         ↓
- Analytical Dataset
- (~12000 rows, ~30 features)
-         ↓
- TimeSeriesSplit CV
-         ↓
- Train models

# snapshot generation

In [1]:
import numpy as np
import pandas as pd

In [2]:
accounts = pd.read_csv(r"C:\Users\AMAN SINGH\Git\sem6miniProject\customer-churn-prediction\data\raw\ravenstack_accounts.csv")
accounts.head(5)

,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,True,False
4,A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,False,True


In [3]:
accounts['signup_date'] = pd.to_datetime(accounts['signup_date'])

# creating the 24 month timeline
snapshot_dates = pd.date_range(start='2023-01-01', end='2024-12-01', freq='MS')

# getting just ids to record the timeline per account
accounts_ids = accounts[['account_id']].copy()
accounts_ids['key'] = 1

# converting the DateTimeIndex to a dataframe to be merged with accounts_Ids
dates_df = pd.DataFrame({'snapshot_date': snapshot_dates})
dates_df['key'] = 1

# meriging ids with the timeline
snapshot_grid = accounts_ids.merge(dates_df, on='key').drop('key', axis=1)

In [4]:
snapshot_grid.head(25) # 24 records per account, the 25th record start of another account

,account_id,snapshot_date
0,A-2e4581,2023-01-01
1,A-2e4581,2023-02-01
2,A-2e4581,2023-03-01
3,A-2e4581,2023-04-01
4,A-2e4581,2023-05-01
5,A-2e4581,2023-06-01
6,A-2e4581,2023-07-01
7,A-2e4581,2023-08-01
8,A-2e4581,2023-09-01
9,A-2e4581,2023-10-01


snapshot_grid
    ↓
merge accounts
    ↓
snapshot (base table)
    ↓
+ subscriptions aggregation
    ↓
+ usage aggregation
    ↓
+ support aggregation
    ↓
FINAL DATASET

### snapshot + accounts 

In [5]:
accounts['signup_date'].min()

Timestamp('2023-01-02 00:00:00')

In [6]:
snapshot_grid['snapshot_date'].min()

Timestamp('2023-01-01 00:00:00')

In [7]:
# merging snapshot_grid with the accounts table
snapshot = snapshot_grid.merge( accounts, on='account_id', how='left')

# keeping only records of months in which the account was existing else no need to keep records of months the account still hasn't signed-up yet
snapshot = snapshot[snapshot['snapshot_date'].dt.to_period('M') >= snapshot['signup_date'].dt.to_period('M')].reset_index(drop=True)

# calculating tenure_days 
"""
tenure_days = 0 → first month of customer
tenure_days > 0 → aging customer)
"""
snapshot['tenure_days'] = (snapshot['snapshot_date'] - snapshot['signup_date']).dt.days.clip(lower=0)

# we keep only static attributes which is same for every month, while other attributes will be aggregated from the rest of the table based on timeline
snapshot = snapshot[[
    'account_id',
    'snapshot_date',
    'industry',
    'country',
    'referral_source',
    'tenure_days'
]]

In [8]:
snapshot.head(6)

,account_id,snapshot_date,industry,country,referral_source,tenure_days
0,A-2e4581,2024-10-01,EdTech,US,partner,0
1,A-2e4581,2024-11-01,EdTech,US,partner,16
2,A-2e4581,2024-12-01,EdTech,US,partner,46
3,A-43a9e3,2023-08-01,FinTech,IN,other,0
4,A-43a9e3,2023-09-01,FinTech,IN,other,15
5,A-43a9e3,2023-10-01,FinTech,IN,other,45


In [9]:
snapshot['snapshot_date'].min()

Timestamp('2023-01-01 00:00:00')

In [10]:
snapshot['account_id'].nunique()

500

In [11]:
snapshot.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5807 entries, 0 to 5806
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   account_id       5807 non-null   object        
 1   snapshot_date    5807 non-null   datetime64[ns]
 2   industry         5807 non-null   object        
 3   country          5807 non-null   object        
 4   referral_source  5807 non-null   object        
 5   tenure_days      5807 non-null   int64         
dtypes: datetime64[ns](1), int64(1), object(4)
memory usage: 272.3+ KB


### snapshot + subscription

In [12]:
subs = pd.read_csv(r"C:\Users\AMAN SINGH\Git\sem6miniProject\customer-churn-prediction\data\raw\ravenstack_subscriptions.csv")
subs.head(5)

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
0,S-8cec59,A-3c1a3f,2023-12-23,2024-04-12,Enterprise,14,2786,33432,False,False,False,True,monthly,True
1,S-0f6f44,A-9b9fe9,2024-06-11,NaN,Pro,17,833,9996,False,False,False,False,monthly,True
2,S-51c0d1,A-659280,2024-11-25,NaN,Enterprise,62,0,0,True,True,False,False,annual,False
3,S-f81687,A-e7a1e2,2024-11-23,2024-12-13,Enterprise,5,995,11940,False,False,False,True,monthly,True
4,S-cff5a2,A-ba6516,2024-01-10,NaN,Enterprise,27,5373,64476,False,False,False,False,monthly,True


In [13]:
subs.shape

(5000, 14)

#### final features decided
total_active_subs,  
total_mrr,  
total_seats,  
max_seats,  
monthly_ratio,  
has_manual_renew,  
active_trials,  
had_downgrade_30,  
had_upgrade_30,  
had_downgrade_31_90,  
had_upgrade_31_90,  
primary_plan,  
enterprise_ratio,  
pro_ratio  

In [14]:
def add_subscription_features(snapshot, subs):

    result = []

    subs['start_date'] = pd.to_datetime(subs['start_date'])
    subs = subs.sort_values(['account_id', 'start_date'])

    for _, row in snapshot.iterrows():
        acc = row['account_id']
        snap_date = row['snapshot_date']

        subs_acc = subs[
            (subs['account_id'] == acc) &
            (subs['start_date'] <= snap_date)
        ]

    #     cols = [
    #     'total_active_subs',
    #     'total_mrr',
    #     'total_seats',
    #     'max_seats',
    #     'monthly_ratio',
    #     'has_manual_renew',
    #     'active_trials',
    #     'had_downgrade_30',
    #     'had_upgrade_30',
    #     'had_downgrade_31_90',
    #     'had_upgrade_31_90',
    #     'primary_plan',
    #     'enterprise_ratio',
    #     'pro_ratio'
    #    ]

        if subs_acc.empty:
            result.append([0]*14)
            continue

        # --- basic ---
        total_active_subs = len(subs_acc)
        total_mrr = subs_acc['mrr_amount'].sum()
        total_seats = subs_acc['seats'].sum()
        max_seats = subs_acc['seats'].max()

        # --- billing ---
        monthly_ratio = (subs_acc['billing_frequency'] == 'monthly').mean().round(3)

        # --- renew ---
        has_manual_renew = (subs_acc['auto_renew_flag'] == False).any()

        # --- trials ---
        active_trials = subs_acc['is_trial'].sum()

        # --- upgrades/downgrades ---
        last_30 = subs_acc[
            subs_acc['start_date'] >= snap_date - pd.Timedelta(days=30)
        ]

        last_31_90 = subs_acc[
            (subs_acc['start_date'] < snap_date - pd.Timedelta(days=30)) &
            (subs_acc['start_date'] >= snap_date - pd.Timedelta(days=90))
        ]

        had_downgrade_30 = last_30['downgrade_flag'].any()
        had_upgrade_30 = last_30['upgrade_flag'].any()

        had_downgrade_31_90 = last_31_90['downgrade_flag'].any()
        had_upgrade_31_90 = last_31_90['upgrade_flag'].any()

        # --- primary plan (most frequent) ---
        primary_plan = subs_acc['plan_tier'].mode().iloc[0]

        # --- enterprise ratio ---
        enterprise_ratio = (subs_acc['plan_tier'] == 'Enterprise').mean().round(3)

        # --- pro ratio ---
        pro_ratio = (subs_acc['plan_tier'] == 'Pro').fillna(0).mean().round(3)

        result.append([
            total_active_subs,
            total_mrr,
            total_seats,
            max_seats,
            monthly_ratio,
            has_manual_renew,
            active_trials,
            had_downgrade_30,
            had_upgrade_30,
            had_downgrade_31_90,
            had_upgrade_31_90,
            primary_plan,
            enterprise_ratio,
            pro_ratio
        ])

    cols = [
        'total_active_subs',
        'total_mrr',
        'total_seats',
        'max_seats',
        'monthly_ratio',
        'has_manual_renew',
        'active_trials',
        'had_downgrade_30',
        'had_upgrade_30',
        'had_downgrade_31_90',
        'had_upgrade_31_90',
        'primary_plan',
        'enterprise_ratio',
        'pro_ratio'
    ]

    features = pd.DataFrame(result, columns=cols)

    return pd.concat([snapshot.reset_index(drop=True), features], axis=1)

In [15]:
snapshot = add_subscription_features(snapshot = snapshot, subs = subs)

In [16]:
subs[subs['account_id'] == 'A-659280'].sort_values('start_date')

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
1606,S-f41a31,A-659280,2023-11-25,NaN,Enterprise,13,2587,31044,False,False,False,False,monthly,False
3998,S-cce4fc,A-659280,2023-11-28,NaN,Pro,13,637,7644,False,False,False,False,annual,True
4737,S-66a834,A-659280,2023-12-23,NaN,Enterprise,31,6169,74028,False,False,False,False,annual,False
2285,S-0b80b4,A-659280,2024-02-08,NaN,Pro,47,2303,27636,False,True,False,False,annual,True
2271,S-96b8d5,A-659280,2024-02-15,NaN,Basic,20,380,4560,False,False,False,False,annual,False
2482,S-3bb8a2,A-659280,2024-04-28,NaN,Enterprise,19,3781,45372,False,False,False,False,monthly,False
488,S-2c4909,A-659280,2024-05-10,NaN,Basic,13,247,2964,False,False,False,False,monthly,False
3383,S-718f18,A-659280,2024-06-05,NaN,Enterprise,24,4776,57312,False,False,False,False,monthly,False
2020,S-dbf124,A-659280,2024-07-25,NaN,Basic,13,247,2964,False,False,False,False,monthly,True
4331,S-4c8e19,A-659280,2024-11-21,NaN,Basic,17,323,3876,False,False,False,False,monthly,False


In [17]:
snapshot[snapshot['account_id'] == 'A-659280']

,account_id,snapshot_date,industry,country,referral_source,tenure_days,total_active_subs,total_mrr,total_seats,max_seats,monthly_ratio,has_manual_renew,active_trials,had_downgrade_30,had_upgrade_30,had_downgrade_31_90,had_upgrade_31_90,primary_plan,enterprise_ratio,pro_ratio
4906,A-659280,2023-11-01,FinTech,IN,event,0,0,0,0,0,0.000,0,0,0,0,0,0,0,0.000,0.000
4907,A-659280,2023-12-01,FinTech,IN,event,14,2,3224,26,13,0.500,True,0,False,False,False,False,Enterprise,0.500,0.500
4908,A-659280,2024-01-01,FinTech,IN,event,45,3,9393,57,31,0.333,True,0,False,False,False,False,Enterprise,0.667,0.333
4909,A-659280,2024-02-01,FinTech,IN,event,76,3,9393,57,31,0.333,True,0,False,False,False,False,Enterprise,0.667,0.333
4910,A-659280,2024-03-01,FinTech,IN,event,105,5,12076,124,47,0.200,True,0,False,True,False,False,Enterprise,0.400,0.400
4911,A-659280,2024-04-01,FinTech,IN,event,136,5,12076,124,47,0.200,True,0,False,False,False,True,Enterprise,0.400,0.400
4912,A-659280,2024-05-01,FinTech,IN,event,166,6,15857,143,47,0.333,True,0,False,False,False,True,Enterprise,0.500,0.333
4913,A-659280,2024-06-01,FinTech,IN,event,197,7,16104,156,47,0.429,True,0,False,False,False,False,Enterprise,0.429,0.286
4914,A-659280,2024-07-01,FinTech,IN,event,227,8,20880,180,47,0.500,True,0,False,False,False,False,Enterprise,0.500,0.250
4915,A-659280,2024-08-01,FinTech,IN,event,258,9,21127,193,47,0.556,True,0,False,False,False,False,Enterprise,0.444,0.222


In [18]:
snapshot.isna().sum()[snapshot.isna().sum() > 0]

Series([], dtype: int64)

### snapshot + feature_usage

In [19]:
usage = pd.read_csv(r"C:\Users\AMAN SINGH\Git\sem6miniProject\customer-churn-prediction\data\raw\ravenstack_feature_usage.csv")
usage.head(3)

,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
0,U-1c6c24,S-0fcf7d,2023-07-27,feature_20,9,5004,0,False
1,U-f07cb8,S-c25263,2023-08-07,feature_5,9,369,0,False
2,U-096807,S-f29e7f,2023-12-07,feature_3,9,1458,0,False


In [20]:
# let's get the account_id in feat_usage 
usage = usage.merge(subs[['subscription_id', 'account_id']], on='subscription_id', how='left')
usage.head(3)

,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature,account_id
0,U-1c6c24,S-0fcf7d,2023-07-27,feature_20,9,5004,0,False,A-e08cd3
1,U-f07cb8,S-c25263,2023-08-07,feature_5,9,369,0,False,A-c7ffc2
2,U-096807,S-f29e7f,2023-12-07,feature_3,9,1458,0,False,A-bbe56f


In [21]:
def add_feature_usage_features(snapshot, usage):

    usage['usage_date'] = pd.to_datetime(usage['usage_date'])
    usage = usage.sort_values(['account_id', 'usage_date'])

    results = []

    for _, row in snapshot.iterrows():
        acc = row['account_id']
        snap_date = row['snapshot_date']

        u = usage[
            (usage['account_id'] == acc) &
            (usage['usage_date'] <= snap_date)
        ]

        if u.empty:
            results.append([0]*9)
            continue

        # --- windows ---
        last_30 = u[
            u['usage_date'] >= snap_date - pd.Timedelta(days=30)
        ]

        last_31_90 = u[
            (u['usage_date'] < snap_date - pd.Timedelta(days=30)) &
            (u['usage_date'] >= snap_date - pd.Timedelta(days=90))
        ]

        # --- recency ---
        last_action_date = u['usage_date'].max()
        days_since_last_action = (snap_date - last_action_date).days

        # --- breadth ---
        unique_30 = last_30['feature_name'].nunique()
        unique_31_90 = last_31_90['feature_name'].nunique()

        breadth_decay_ratio = (round(unique_30 / unique_31_90, 3) if unique_31_90 > 0 else 0)

        # --- frequency ---
        active_days_30d = last_30['usage_date'].nunique()
        active_days_31_to_90d = last_31_90['usage_date'].nunique()
        total_usage_count_30d = last_30['usage_count'].sum()

        activity_drop_ratio = (round(active_days_30d / active_days_31_to_90d, 3) if active_days_31_to_90d > 0 else 0)

        avg_duration_30 = (round(last_30['usage_duration_secs'].mean(), 3) if not last_30.empty else 0)

        # --- errors ---
        had_error_30 = (last_30['error_count'] > 0).any()

        results.append([
            days_since_last_action,
            unique_30,
            unique_31_90,
            breadth_decay_ratio,
            active_days_30d,
            activity_drop_ratio,
            total_usage_count_30d,
            avg_duration_30,
            int(had_error_30)
        ])

    cols = [
        'days_since_last_action',
        'unique_features_last_30d',
        'unique_features_31_to_90d',
        'breadth_decay_ratio',
        'active_days_last_30d',
        'activity_drop_ratio',
        'total_usage_count_30d',
        'avg_duration_last_30d',
        'had_error_last_30d'
    ]

    features = pd.DataFrame(results, columns=cols)

    return pd.concat([snapshot.reset_index(drop=True), features], axis=1)

In [22]:
snapshot = add_feature_usage_features(snapshot = snapshot, usage = usage)

In [23]:
snapshot[snapshot['account_id'] == 'A-659280']

,account_id,snapshot_date,industry,country,referral_source,tenure_days,total_active_subs,total_mrr,total_seats,max_seats,...,pro_ratio,days_since_last_action,unique_features_last_30d,unique_features_31_to_90d,breadth_decay_ratio,active_days_last_30d,activity_drop_ratio,total_usage_count_30d,avg_duration_last_30d,had_error_last_30d
4906,A-659280,2023-11-01,FinTech,IN,event,0,0,0,0,0,...,0.000,1,5,1,5.000,5,5.000,54,3509.200,1
4907,A-659280,2023-12-01,FinTech,IN,event,14,2,3224,26,13,...,0.500,11,3,6,0.500,2,0.333,36,3904.000,0
4908,A-659280,2024-01-01,FinTech,IN,event,45,3,9393,57,31,...,0.333,27,1,8,0.125,1,0.143,6,2574.000,0
4909,A-659280,2024-02-01,FinTech,IN,event,76,3,9393,57,31,...,0.333,13,3,4,0.750,3,1.000,29,2198.667,1
4910,A-659280,2024-03-01,FinTech,IN,event,105,5,12076,124,47,...,0.400,42,0,4,0.000,0,0.000,0,0.000,0
4911,A-659280,2024-04-01,FinTech,IN,event,136,5,12076,124,47,...,0.400,3,2,3,0.667,2,0.667,14,1956.500,1
4912,A-659280,2024-05-01,FinTech,IN,event,166,6,15857,143,47,...,0.333,4,6,2,3.000,6,3.000,78,3654.875,1
4913,A-659280,2024-06-01,FinTech,IN,event,197,7,16104,156,47,...,0.286,12,2,7,0.286,2,0.250,21,2386.500,1
4914,A-659280,2024-07-01,FinTech,IN,event,227,8,20880,180,47,...,0.250,18,1,8,0.125,1,0.125,5,1580.000,0
4915,A-659280,2024-08-01,FinTech,IN,event,258,9,21127,193,47,...,0.222,30,1,3,0.333,1,0.333,13,2717.000,0


### snapshot + support_tickets

tickets_last_30d  
tickets_31_to_90d  
escalated_last_90d  
had_urgent_ticket_last_90d  
avg_resolution_last_90d  
latest_csat_score  

In [24]:
sup_ticks = pd.read_csv(r"C:\Users\AMAN SINGH\Git\sem6miniProject\customer-churn-prediction\data\raw\ravenstack_support_tickets.csv")
sup_ticks.head(3)

,ticket_id,account_id,submitted_at,closed_at,resolution_time_hours,priority,first_response_time_minutes,satisfaction_score,escalation_flag
0,T-0024de,A-712f1c,2023-07-27,2023-07-28 03:00:00,27.0,high,74,NaN,False
1,T-4d04b9,A-e43bf7,2024-07-08,2024-07-09 03:00:00,27.0,urgent,144,NaN,False
2,T-d5e12f,A-0f3e88,2024-10-17,2024-10-17 19:00:00,19.0,urgent,93,4.0,False


In [25]:
sup_ticks[sup_ticks['account_id'] == 'A-659280']

,ticket_id,account_id,submitted_at,closed_at,resolution_time_hours,priority,first_response_time_minutes,satisfaction_score,escalation_flag
42,T-6cce32,A-659280,2024-06-12,2024-06-12 06:00:00,6.0,medium,176,4.0,False
889,T-9fcefd,A-659280,2023-04-17,2023-04-17 02:00:00,2.0,low,91,4.0,False
919,T-0ebaee,A-659280,2024-03-05,2024-03-05 02:00:00,2.0,medium,178,NaN,False
1404,T-b55b60,A-659280,2024-08-25,2024-08-25 22:00:00,22.0,low,50,5.0,False


In [26]:
def add_support_ticket_features(snapshot, tickets):

    tickets['submitted_at'] = pd.to_datetime(tickets['submitted_at'])
    tickets = tickets.sort_values(['account_id', 'submitted_at'])

    results = []

    for _, row in snapshot.iterrows():
        acc = row['account_id']
        snap_date = row['snapshot_date']

        t = tickets[
            (tickets['account_id'] == acc) &
            (tickets['submitted_at'] <= snap_date)
        ]

        if t.empty:
            results.append([0]*6)
            continue

        # --- windows ---
        last_30 = t[
            t['submitted_at'] >= snap_date - pd.Timedelta(days=30)
        ]

        last_31_90 = t[
            (t['submitted_at'] < snap_date - pd.Timedelta(days=30)) &
            (t['submitted_at'] >= snap_date - pd.Timedelta(days=90))
        ]

        last_90 = t[
            t['submitted_at'] >= snap_date - pd.Timedelta(days=90)
        ]

        # --- counts ---
        tickets_30 = len(last_30)
        tickets_31_90 = len(last_31_90)

        # --- escalation ---
        escalated_90 = (last_90['escalation_flag'] == True).any()

        # --- urgent ---
        urgent_90 = (last_90['priority'] == 'urgent').any()

        # --- resolution ---
        avg_resolution_90 = (round(last_90['resolution_time_hours'].mean(), 3) if not last_90.empty else 0)

        # --- csat ---
        latest_ticket = t.sort_values('submitted_at').iloc[-1]
        latest_csat = latest_ticket['satisfaction_score']

        if pd.isna(latest_csat):
            acc_csat = t['satisfaction_score'].dropna()
            latest_csat = acc_csat.mode().iloc[0] if not acc_csat.empty else tickets['satisfaction_score'].median()

        results.append([
            tickets_30,
            tickets_31_90,
            int(escalated_90),
            int(urgent_90),
            avg_resolution_90,
            latest_csat
        ])

    cols = [
        'tickets_last_30d',
        'tickets_31_to_90d',
        'escalated_last_90d',
        'had_urgent_ticket_last_90d',
        'avg_resolution_last_90d',
        'latest_csat_score'
    ]

    features = pd.DataFrame(results, columns=cols)

    return pd.concat([snapshot.reset_index(drop=True), features], axis=1)

In [27]:
snapshot = add_support_ticket_features(snapshot=snapshot, tickets=sup_ticks)

In [28]:
snapshot[snapshot['account_id'] == 'A-659280']

,account_id,snapshot_date,industry,country,referral_source,tenure_days,total_active_subs,total_mrr,total_seats,max_seats,...,activity_drop_ratio,total_usage_count_30d,avg_duration_last_30d,had_error_last_30d,tickets_last_30d,tickets_31_to_90d,escalated_last_90d,had_urgent_ticket_last_90d,avg_resolution_last_90d,latest_csat_score
4906,A-659280,2023-11-01,FinTech,IN,event,0,0,0,0,0,...,5.000,54,3509.200,1,0,0,0,0,0.0,4.0
4907,A-659280,2023-12-01,FinTech,IN,event,14,2,3224,26,13,...,0.333,36,3904.000,0,0,0,0,0,0.0,4.0
4908,A-659280,2024-01-01,FinTech,IN,event,45,3,9393,57,31,...,0.143,6,2574.000,0,0,0,0,0,0.0,4.0
4909,A-659280,2024-02-01,FinTech,IN,event,76,3,9393,57,31,...,1.000,29,2198.667,1,0,0,0,0,0.0,4.0
4910,A-659280,2024-03-01,FinTech,IN,event,105,5,12076,124,47,...,0.000,0,0.000,0,0,0,0,0,0.0,4.0
4911,A-659280,2024-04-01,FinTech,IN,event,136,5,12076,124,47,...,0.667,14,1956.500,1,1,0,0,0,2.0,4.0
4912,A-659280,2024-05-01,FinTech,IN,event,166,6,15857,143,47,...,3.000,78,3654.875,1,0,1,0,0,2.0,4.0
4913,A-659280,2024-06-01,FinTech,IN,event,197,7,16104,156,47,...,0.250,21,2386.500,1,0,1,0,0,2.0,4.0
4914,A-659280,2024-07-01,FinTech,IN,event,227,8,20880,180,47,...,0.125,5,1580.000,0,1,0,0,0,6.0,4.0
4915,A-659280,2024-08-01,FinTech,IN,event,258,9,21127,193,47,...,0.333,13,2717.000,0,0,1,0,0,6.0,4.0


In [29]:
snapshot['account_id'].nunique()

500

### adding the target label

In [30]:
churn_events = pd.read_csv(r"C:\Users\AMAN SINGH\Git\sem6miniProject\customer-churn-prediction\data\raw\ravenstack_churn_events.csv")
churn_events.head(3)

,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text
0,C-816288,A-c37cab,2024-10-27,pricing,4.03,False,False,False,switched to competitor
1,C-5a81e7,A-37f969,2024-06-25,support,96.45,True,False,False,NaN
2,C-a174be,A-b07346,2024-11-12,budget,0.00,False,False,False,missing features


In [31]:
def add_target_label(snapshot, churn):

    snapshot = snapshot.copy()
    churn = churn.copy()

    snapshot['snapshot_date'] = pd.to_datetime(snapshot['snapshot_date'])
    churn['churn_date'] = pd.to_datetime(churn['churn_date'])

    snapshot = snapshot.sort_values(['snapshot_date', 'account_id'])
    churn = churn.sort_values(['churn_date', 'account_id'])

    # merge next churn
    merged = pd.merge_asof(
        snapshot,
        churn[['account_id', 'churn_date']],
        by='account_id',
        left_on='snapshot_date',
        right_on='churn_date',
        direction='forward'
    )

    # compute days
    merged['days_until_next_churn'] = (
        merged['churn_date'] - merged['snapshot_date']
    ).dt.days

    # target
    merged['target'] = (
        (merged['days_until_next_churn'] >= 0) &
        (merged['days_until_next_churn'] <= 30)
    ).astype(int)

    # cleanup
    final = merged.drop(columns=['churn_date', 'days_until_next_churn'])

    return final

In [32]:
snapshot = add_target_label(snapshot=snapshot, churn=churn_events)

In [33]:
snapshot[snapshot['account_id'] == 'A-659280']

,account_id,snapshot_date,industry,country,referral_source,tenure_days,total_active_subs,total_mrr,total_seats,max_seats,...,total_usage_count_30d,avg_duration_last_30d,had_error_last_30d,tickets_last_30d,tickets_31_to_90d,escalated_last_90d,had_urgent_ticket_last_90d,avg_resolution_last_90d,latest_csat_score,target
1067,A-659280,2023-11-01,FinTech,IN,event,0,0,0,0,0,...,54,3509.200,1,0,0,0,0,0.0,4.0,0
1281,A-659280,2023-12-01,FinTech,IN,event,14,2,3224,26,13,...,36,3904.000,0,0,0,0,0,0.0,4.0,0
1516,A-659280,2024-01-01,FinTech,IN,event,45,3,9393,57,31,...,6,2574.000,0,0,0,0,0,0.0,4.0,0
1760,A-659280,2024-02-01,FinTech,IN,event,76,3,9393,57,31,...,29,2198.667,1,0,0,0,0,0.0,4.0,0
2029,A-659280,2024-03-01,FinTech,IN,event,105,5,12076,124,47,...,0,0.000,0,0,0,0,0,0.0,4.0,0
2321,A-659280,2024-04-01,FinTech,IN,event,136,5,12076,124,47,...,14,1956.500,1,1,0,0,0,2.0,4.0,0
2638,A-659280,2024-05-01,FinTech,IN,event,166,6,15857,143,47,...,78,3654.875,1,0,1,0,0,2.0,4.0,0
2972,A-659280,2024-06-01,FinTech,IN,event,197,7,16104,156,47,...,21,2386.500,1,0,1,0,0,2.0,4.0,0
3327,A-659280,2024-07-01,FinTech,IN,event,227,8,20880,180,47,...,5,1580.000,0,1,0,0,0,6.0,4.0,1
3710,A-659280,2024-08-01,FinTech,IN,event,258,9,21127,193,47,...,13,2717.000,0,0,1,0,0,6.0,4.0,0


In [34]:
churn_events[churn_events['account_id'] == 'A-659280']

,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text
77,C-bb9db4,A-659280,2024-11-22,pricing,27.75,False,False,False,switched to competitor
240,C-921335,A-659280,2024-07-23,features,0.00,False,False,False,too expensive


In [35]:
snapshot['snapshot_date'].min()

Timestamp('2023-01-01 00:00:00')

### removing 'ghost rows'

removing the month rows where the account was inactive for the whole month

In [36]:
snapshot = snapshot[
    (snapshot['total_mrr'] > 0) |
    (snapshot['active_trials'] > 0) |
    (snapshot['total_usage_count_30d'] > 0) |
    (snapshot['tickets_last_30d'] > 0)
].reset_index(drop=True)

In [37]:
snapshot.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5646 entries, 0 to 5645
Data columns (total 36 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   account_id                  5646 non-null   object        
 1   snapshot_date               5646 non-null   datetime64[ns]
 2   industry                    5646 non-null   object        
 3   country                     5646 non-null   object        
 4   referral_source             5646 non-null   object        
 5   tenure_days                 5646 non-null   int64         
 6   total_active_subs           5646 non-null   int64         
 7   total_mrr                   5646 non-null   int64         
 8   total_seats                 5646 non-null   int64         
 9   max_seats                   5646 non-null   int64         
 10  monthly_ratio               5646 non-null   float64       
 11  has_manual_renew            5646 non-null   object      

In [38]:
## to see all the columns in the output
# with pd.option_context('display.max_columns', None, 'display.width', None):
#     display(snapshot[
#     (snapshot['total_mrr'] > 0) |
#     (snapshot['active_trials'] > 0) |
#     (snapshot['total_usage_count_30d'] > 0) |
#     (snapshot['tickets_last_30d'] > 0)])


In [39]:
snapshot['snapshot_date'].min()

Timestamp('2023-02-01 00:00:00')

In [40]:
snapshot['account_id'].nunique()

498

In [41]:
snapshot.isna().sum()[snapshot.isna().sum() > 0]

Series([], dtype: int64)

In [42]:
accounts[accounts['account_id'] == 'A-10b8f2']

,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
151,A-10b8f2,Company_151,HealthTech,DE,2023-01-11,organic,Basic,47,True,False


In [43]:
subs[subs['account_id'] == 'A-10b8f2']

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
31,S-49a9aa,A-10b8f2,2023-06-16,NaN,Basic,51,969,11628,False,False,False,False,annual,True
425,S-d4cc9b,A-10b8f2,2023-10-03,NaN,Pro,54,2646,31752,False,False,False,False,annual,True
843,S-5e558a,A-10b8f2,2023-10-07,NaN,Enterprise,47,9353,112236,False,False,False,False,annual,True
1270,S-bff3b5,A-10b8f2,2024-03-22,NaN,Pro,47,2303,27636,False,False,False,False,monthly,False
1588,S-3ff7bb,A-10b8f2,2024-08-15,NaN,Basic,47,0,0,True,True,False,False,monthly,True
1864,S-7ea264,A-10b8f2,2024-04-10,NaN,Enterprise,47,9353,112236,False,False,False,False,annual,True
1969,S-4a2fc6,A-10b8f2,2023-08-05,NaN,Basic,47,0,0,True,False,False,False,annual,True
2856,S-4a89a7,A-10b8f2,2023-04-19,NaN,Basic,47,893,10716,False,False,False,False,monthly,False
3155,S-1cefc9,A-10b8f2,2024-12-16,NaN,Pro,48,2352,28224,False,False,False,False,annual,True
3687,S-fa9b43,A-10b8f2,2023-02-05,NaN,Basic,47,0,0,True,False,False,False,annual,False


In [44]:
# usage[usage['account_id'] == 'A-10b8f2'].sort_values('usage_date')

In [45]:
snapshot.columns

Index(['account_id', 'snapshot_date', 'industry', 'country', 'referral_source',
       'tenure_days', 'total_active_subs', 'total_mrr', 'total_seats',
       'max_seats', 'monthly_ratio', 'has_manual_renew', 'active_trials',
       'had_downgrade_30', 'had_upgrade_30', 'had_downgrade_31_90',
       'had_upgrade_31_90', 'primary_plan', 'enterprise_ratio', 'pro_ratio',
       'days_since_last_action', 'unique_features_last_30d',
       'unique_features_31_to_90d', 'breadth_decay_ratio',
       'active_days_last_30d', 'activity_drop_ratio', 'total_usage_count_30d',
       'avg_duration_last_30d', 'had_error_last_30d', 'tickets_last_30d',
       'tickets_31_to_90d', 'escalated_last_90d', 'had_urgent_ticket_last_90d',
       'avg_resolution_last_90d', 'latest_csat_score', 'target'],
      dtype='object')

In [46]:
len(snapshot.columns)

36

In [47]:
snapshot['had_downgrade_30'].unique(), snapshot['had_upgrade_30'].unique(), snapshot['had_downgrade_31_90'].unique(), snapshot['had_upgrade_31_90'].unique()

(array([0, np.True_], dtype=object),
 array([0, np.True_], dtype=object),
 array([0, np.True_], dtype=object),
 array([0, np.True_], dtype=object))

In [109]:
test_df = pd.read_csv(r"C:\Users\AMAN SINGH\Git\sem6miniProject\customer-churn-prediction\data\processed\test_df.csv")    

In [110]:
test_df.shape

(1831, 36)

In [111]:
test21_df = pd.read_csv(r"C:\Users\AMAN SINGH\Git\sem6miniProject\customer-churn-prediction\data\processed\test21_df.csv")
test21_df.shape

(416, 37)

In [125]:
import joblib

# Load the saved feature list
model_features = joblib.load(r"C:\Users\AMAN SINGH\Git\sem6miniProject\customer-churn-prediction\models\model_features_rf1.pkl")

In [99]:
print(model_features)

['tenure_days', 'total_active_subs', 'total_mrr', 'total_seats', 'max_seats', 'monthly_ratio', 'active_trials', 'enterprise_ratio', 'pro_ratio', 'days_since_last_action', 'unique_features_last_30d', 'unique_features_31_to_90d', 'breadth_decay_ratio', 'active_days_last_30d', 'activity_drop_ratio', 'total_usage_count_30d', 'avg_duration_last_30d', 'had_error_last_30d', 'tickets_last_30d', 'tickets_31_to_90d', 'escalated_last_90d', 'had_urgent_ticket_last_90d', 'avg_resolution_last_90d', 'latest_csat_score', 'industry_DevTools', 'industry_EdTech', 'industry_FinTech', 'industry_HealthTech', 'country_CA', 'country_DE', 'country_FR', 'country_IN', 'country_UK', 'country_US', 'referral_source_event', 'referral_source_organic', 'referral_source_other', 'referral_source_partner', 'has_manual_renew_False', 'has_manual_renew_True', 'had_downgrade_30_False', 'had_downgrade_30_True', 'had_upgrade_30_False', 'had_upgrade_30_True', 'had_downgrade_31_90_False', 'had_downgrade_31_90_True', 'had_upgrade

In [126]:
print("Number of features:", len(model_features))

Number of features: 60


In [88]:
# Columns in your current DataFrame
df_columns = dummy1.columns.tolist()

# Compare sets
missing_in_df = set(model_features) - set(df_columns)
extra_in_df   = set(df_columns) - set(model_features)

print("Missing features:", missing_in_df)
print("Unexpected extra features:", extra_in_df)

Missing features: set()
Unexpected extra features: {'had_upgrade_31_90_0', 'industry_Cybersecurity', 'had_upgrade_30_0', 'referral_source_ads', 'country_AU', 'had_downgrade_30_0', 'has_manual_renew_0', 'primary_plan_0', 'had_downgrade_31_90_0'}


In [61]:
# snapshot.to_csv(r"C:\Users\AMAN SINGH\Git\sem6miniProject\customer-churn-prediction\data\processed\final_dataset.csv", index=False)

### train/test split

In [62]:
# cutoff_date = snapshot['snapshot_date'].max() - pd.DateOffset(months=6)
# train_df1 = snapshot[snapshot['snapshot_date'] <= cutoff_date]
# test_df1  = snapshot[snapshot['snapshot_date'] > cutoff_date]

In [63]:
# train_df1.shape, test_df1.shape

In [64]:
# test_df1.info()

In [65]:
# snapshot.isna().sum()[snapshot.isna().sum() > 0]

In [ ]:
# months = sorted(snapshot['snapshot_date'].unique())
# test_months = months[-4:]
# train_df = snapshot[~snapshot['snapshot_date'].isin(test_months)]
# test_df  = snapshot[snapshot['snapshot_date'].isin(test_months)]

In [71]:
# train_df.shape, test_df.shape

In [ ]:
# train_df.to_csv(r"C:\Users\AMAN SINGH\Git\sem6miniProject\customer-churn-prediction\data\processed\train_df.csv", index=False)
# test_df.to_csv(r"C:\Users\AMAN SINGH\Git\sem6miniProject\customer-churn-prediction\data\processed\test_df.csv", index=False)

splitting test_df (6 months - 19-24) to 6 dataframes of each month  
19th will be used for test_df for best model evaluation  
20-24 will simulate monthly batch arrival  

In [6]:
import pandas as pd

In [7]:
test_df = pd.read_csv(r"C:\Users\AMAN SINGH\Git\sem6miniProject\customer-churn-prediction\data\processed\not-inference\test_df.csv")

In [8]:
test_df['snapshot_date'] = pd.to_datetime(test_df['snapshot_date'])

In [9]:
test_df['year_month'] = test_df['snapshot_date'].dt.to_period('M')

In [10]:
test_df.sample(5)

,account_id,snapshot_date,industry,country,referral_source,tenure_days,total_active_subs,total_mrr,total_seats,max_seats,...,avg_duration_last_30d,had_error_last_30d,tickets_last_30d,tickets_31_to_90d,escalated_last_90d,had_urgent_ticket_last_90d,avg_resolution_last_90d,latest_csat_score,target,year_month
251,A-98a59a,2024-09-01,FinTech,UK,partner,394,8,22568,540,148,...,792.000,0,0,0,0,0,0.0,4.0,1,2024-09
735,A-bb3bd4,2024-10-01,HealthTech,US,ads,248,5,7015,115,23,...,2940.000,1,2,0,0,1,27.5,5.0,0,2024-10
1636,A-9a532a,2024-12-01,HealthTech,US,partner,148,4,10945,117,62,...,0.000,0,0,0,0,0,0.0,4.0,0,2024-12
829,A-f03140,2024-10-01,DevTools,IN,event,314,8,15702,228,38,...,5673.500,0,0,0,0,0,0.0,5.0,1,2024-10
420,A-019782,2024-10-01,DevTools,FR,event,531,8,8353,108,17,...,3636.667,1,1,0,0,1,10.0,4.0,0,2024-10


In [11]:
monthly_test_dfs = {
    str(month): df.drop(columns='year_month').reset_index(drop=True) for month, df in test_df.groupby('year_month')
}

In [12]:
monthly_test_dfs.keys()

dict_keys(['2024-09', '2024-10', '2024-11', '2024-12'])

In [13]:
test21_df = monthly_test_dfs['2024-09']
test22_df = monthly_test_dfs['2024-10']
test23_df = monthly_test_dfs['2024-11']
test24_df = monthly_test_dfs['2024-12']
# display(test21_df.shape)
# display(test21_df.head(5))

In [14]:
display(test24_df['snapshot_date'].min())
display(test24_df['snapshot_date'].max())

Timestamp('2024-12-01 00:00:00')

Timestamp('2024-12-01 00:00:00')

In [15]:
test21_df.to_csv(r"C:\Users\AMAN SINGH\Git\sem6miniProject\customer-churn-prediction\data\processed\inference\inference2\september_2024.csv", index=False)
test22_df.to_csv(r"C:\Users\AMAN SINGH\Git\sem6miniProject\customer-churn-prediction\data\processed\inference\inference2\october_2024.csv", index=False)
test23_df.to_csv(r"C:\Users\AMAN SINGH\Git\sem6miniProject\customer-churn-prediction\data\processed\inference\inference2\november_2024.csv", index=False)
test24_df.to_csv(r"C:\Users\AMAN SINGH\Git\sem6miniProject\customer-churn-prediction\data\processed\inference\inference2\december_2024.csv", index=False)

In [ ]:
import pandas as pd
import os

# Point this to your CSV file
file_path = r'C:\Users\AMAN SINGH\Git\sem6miniProject\customer-churn-prediction\data\processed\inference\september_2024.csv'

print("⏳ Reading corrupted CSV...")
df = pd.read_csv(file_path)
print(df.head()[:10])
# Find and drop any column that Pandas auto-named 'Unnamed: 0'
cols_to_drop = [c for c in df.columns if 'Unnamed' in str(c)]
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"🗑️ Dropped phantom index columns: {cols_to_drop}")
print(df.head()[:10])
# Save it back exactly the way it should be (THIS prevents it from happening again)
df.to_csv(file_path, index=False)
print("✅ CSV cleaned and successfully overwritten!")

In [5]:
import pandas as pd

file_path = r'C:\Users\AMAN SINGH\Git\sem6miniProject\customer-churn-prediction\data\processed\inference\september_2024.csv'


# index_col=0 tells Pandas: "The very first column is just the row numbers. Absorb it as the index."
df = pd.read_csv(file_path, index_col=0)
print(df.head(10)[:10])
# index=False tells Pandas: "When you save, DO NOT write that index back into the file."
df.to_csv(file_path, index=False)

print("✅ CSV perfectly cleaned!")

              country referral_source  tenure_days  total_active_subs  \
industry                                                                
Cybersecurity      US             ads          292                 10   
HealthTech         AU         organic          352                  6   
FinTech            DE         partner          102                  3   
EdTech             CA         organic           32                  1   
DevTools           FR           event          501                  8   
Cybersecurity      CA           event          178                  5   
HealthTech         IN           other          157                 12   
FinTech            US           event          514                  6   
EdTech             US           event          301                  2   
DevTools           FR           other          257                  2   

               total_mrr  total_seats  max_seats  monthly_ratio  \
industry                                                